# Configuración de Google BigQuery

La función de este notebook es configurar la conexión con Google Cloud y crear el dataset necesario para el proyecto de la tienda de artículos electrónicos **SkeletIA**.

In [1]:
# Comprobación desde dónde se ejecuta el notebook
from pathlib import Path

Path.cwd()

PosixPath('/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/parte_2_modelo_bigquery/notebooks')

In [2]:
# Para evitar problemas añado como raíz la carpeta desde la que se ejecuta
# Si no encuentra ahí .env, que suba dos niveles
from pathlib import Path
from dotenv import load_dotenv
import os

ROOT_DIR = Path.cwd()

if not (ROOT_DIR / ".env").exists():
    ROOT_DIR = ROOT_DIR.parent.parent

ENV_PATH = ROOT_DIR / ".env"

load_dotenv(ENV_PATH)

# Conversión de ruta relativa de credentials a absoluta
credentials_path = Path(
    os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
)

if not credentials_path.is_absolute():
    credentials_path = ROOT_DIR / credentials_path

credentials_path = credentials_path.resolve()

# Actualización de la variable de entorno
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path)

In [3]:
# Cargar las credenciales en una variable
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

In [4]:
# Crear cliente de BigQuery
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

## Creación del Dataset
Ahora que ya tenemos el cliente se procede a crear el `Dataset`.  
Dentro del `Dataset` van a ir las tablas

In [5]:
# Primero creo el objeto en Python
dataset_full_id = f"{PROJECT_ID}.{DATASET_ID}"
dataset = bigquery.Dataset(dataset_full_id)

# Si la empresa fuera internacional mejor incluir de dónde son estos datos
dataset.location = "EU"

In [6]:
# Creación del Dataset en Google Cloud
dataset = client.create_dataset(
    dataset,
    exists_ok=True # para evitar errores por múltiples ejecuciones
)

# Comprobación de que ya existe
dataset_ref = client.get_dataset(dataset_full_id)

print("Dataset:", dataset_ref.dataset_id)
print("Proyecto:", dataset_ref.project)
print("Ubicación:", dataset_ref.location)

Dataset: skeletia
Proyecto: tc-sql-bometon
Ubicación: EU


## Creación de las tablas maestras o de referencia

Las **tablas maestras** son aquellas que son relativamente estables y otras tablas las referencian.  
Estas son las primeras que se crean para que no haya errores en su declaración.

1. `countries`
2. `cities`
3. `acquisition_channels`
4. `categories`
5. `brands`

Las claves primarias (PK) y foráneas (FK) se declaran como `NOT ENFORCED`, ya que BigQuery almacena estas restricciones pero no garantiza su cumplimiento.  
En BigQuery `NOT ENFORCED` es **obligatorio**. En SQL sí se admite `ENFORCED`. BigQuery no va a comprobar que la PK sea realmente única, ni que una FK exista realmente en la tabla padre.  
Las restricciones `UNIQUE` y `CHECK` del modelo MySQL se validarán durante la generación y carga de los datos.

En primer lugar creo una función básica para hacer todo el proceso de **Data Definition Language** (DDL).  
Se encargará de enviar la intrucción a BigQuery y esperar a que termine antes de seguir con la siguiente tabla.

In [7]:
def ejecutar_ddl(sql):
    """
    Ejecuta una sentencia DDL en BigQuery y espera a que termine.
    """
    job = client.query(sql)
    job.result()

BigQuery no tiene `AUTO_INCREMENT`. Los identificadores de las PK se van a generar desde Python.

In [8]:
sql_countries = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.countries` (
    country_id INT64 NOT NULL,
    country_code STRING(2) NOT NULL,
    country_name STRING(80) NOT NULL,

    PRIMARY KEY (country_id) NOT ENFORCED
)
"""

ejecutar_ddl(sql_countries)

In [9]:
sql_cities = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.cities` (
    city_id INT64 NOT NULL,
    country_id INT64 NOT NULL,
    city_name STRING(100) NOT NULL,

    PRIMARY KEY (city_id) NOT ENFORCED,

    CONSTRAINT fk_cities_country
        FOREIGN KEY (country_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.countries`(country_id)
        NOT ENFORCED
)
"""

ejecutar_ddl(sql_cities)

In [10]:
sql_acquisition_channels = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.acquisition_channels` (
    channel_id INT64 NOT NULL,
    channel_code STRING(30) NOT NULL,
    channel_name STRING(80) NOT NULL,

    PRIMARY KEY (channel_id) NOT ENFORCED
)
"""

ejecutar_ddl(sql_acquisition_channels)

In [11]:
sql_categories = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.categories` (
    category_id INT64 NOT NULL,
    category_name STRING(80) NOT NULL,
    description STRING(255),

    PRIMARY KEY (category_id) NOT ENFORCED
)
"""

ejecutar_ddl(sql_categories)

In [12]:
sql_brands = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.brands` (
    brand_id INT64 NOT NULL,
    brand_name STRING(80) NOT NULL,

    PRIMARY KEY (brand_id) NOT ENFORCED
)
"""

ejecutar_ddl(sql_brands)

### Comprobaciones sobre las tablas maestras o de referencia

In [13]:
# Comprobación de que las tablas de referencia se han creado correctamente
tablas = list(client.list_tables(DATASET_ID))

print("Tablas existentes:")

for t in tablas:
    print("-", t.table_id)

Tablas existentes:
- acquisition_channels
- brands
- categories
- cities
- countries
- customers
- order_items
- orders
- payments
- products
- reviews


In [14]:
sql_constraints = f"""
SELECT
    table_name,
    constraint_name,
    constraint_type,
    enforced
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLE_CONSTRAINTS`
ORDER BY table_name, constraint_type
"""

constraints = client.query(sql_constraints).to_dataframe()

constraints

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,constraint_name,constraint_type,enforced
0,acquisition_channels,acquisition_channels.pk$,PRIMARY KEY,NO
1,brands,brands.pk$,PRIMARY KEY,NO
2,categories,categories.pk$,PRIMARY KEY,NO
3,cities,cities.fk_cities_country,FOREIGN KEY,NO
4,cities,cities.pk$,PRIMARY KEY,NO
5,countries,countries.pk$,PRIMARY KEY,NO
6,customers,customers.fk_customers_city,FOREIGN KEY,NO
7,customers,customers.fk_customers_channel,FOREIGN KEY,NO
8,customers,customers.pk$,PRIMARY KEY,NO
9,order_items,order_items.fk_order_items_order,FOREIGN KEY,NO


## Creación de las tablas de negocio

Estas son las tablas que representan las entidades y operaciones del negocio.

1. `customers`
2. `products`

Es necesario crear primero esas dos tablas para continuar con las tablas transaccionales, que dependen de ellas.

1. `orders`
2. `order_items`
3. `payments`
4. `reviews`

BigQuery no tiene implementada la restricción `UNIQUE` como sí la tienen otros clientes.  
Es necesario para que el atributo `email` sea un valor único como exige el enunciado.  
Se hará esa restricción desde Python en el momento de generar y cargar los datos.

In [15]:
sql_customers = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.customers` (
    customer_id INT64 NOT NULL,
    first_name STRING(80) NOT NULL,
    last_name STRING(120) NOT NULL,
    email STRING(180) NOT NULL,
    phone STRING(30),
    city_id INT64 NOT NULL,
    channel_id INT64 NOT NULL,
    registered_at DATETIME NOT NULL,
    is_active BOOL DEFAULT TRUE NOT NULL,

    PRIMARY KEY (customer_id) NOT ENFORCED,

    CONSTRAINT fk_customers_city
        FOREIGN KEY (city_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.cities`(city_id)
        NOT ENFORCED,

    CONSTRAINT fk_customers_channel
        FOREIGN KEY (channel_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.acquisition_channels`(channel_id)
        NOT ENFORCED
)
"""

ejecutar_ddl(sql_customers)

Otra diferencia entre otros clientes de SQL y BigQuery es `NUMERIC`.  
En otros clientes sería `DECIMAL`.

In [16]:
sql_products = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.products` (
    product_id INT64 NOT NULL,
    sku STRING(40) NOT NULL,
    category_id INT64 NOT NULL,
    brand_id INT64 NOT NULL,
    product_name STRING(180) NOT NULL,
    current_sale_price NUMERIC(10, 2) NOT NULL,
    current_cost NUMERIC(10, 2) NOT NULL,
    stock INT64 DEFAULT 0 NOT NULL,
    is_active BOOL DEFAULT TRUE NOT NULL,
    created_at DATETIME DEFAULT CURRENT_DATETIME() NOT NULL,

    PRIMARY KEY (product_id) NOT ENFORCED,

    CONSTRAINT fk_products_category
        FOREIGN KEY (category_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.categories`(category_id)
        NOT ENFORCED,

    CONSTRAINT fk_products_brand
        FOREIGN KEY (brand_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.brands`(brand_id)
        NOT ENFORCED
)
"""

ejecutar_ddl(sql_products)

### Creación de las tablas transaccionales

Estas son las que dependen de las tablas de negocio.

En la tabla `orders` se almacenan los pedidos realizados por los clientes.

Para el atributo `status` sería conveniente utilizar algo así como `ENUM`.  
BigQuery no tiene una solución para ello. Así que esa validación se hará desde Python con sus posibilidades:

```python
VALID_ORDER_STATUSES = {
    "cancelled",
    "confirmed",
    "delivered",
    "pending",
    "returned",
    "shipped",    
}
```

Se tendrá que hacer una comprobación antes de hacer la inserción a la base de datos.

In [17]:
sql_orders = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.orders` (
    order_id INT64 NOT NULL,
    customer_id INT64 NOT NULL,

    status STRING DEFAULT 'pending' NOT NULL,

    order_date DATETIME NOT NULL,
    shipped_at DATETIME,
    delivered_at DATETIME,

    shipping_recipient STRING(200) NOT NULL,
    shipping_address_line1 STRING(200) NOT NULL,
    shipping_postal_code STRING(20) NOT NULL,
    shipping_city_id INT64 NOT NULL,

    shipping_cost NUMERIC(8, 2) DEFAULT 0.00 NOT NULL,
    currency_code STRING(3) DEFAULT 'EUR' NOT NULL,

    PRIMARY KEY (order_id) NOT ENFORCED,

    CONSTRAINT fk_orders_customer
        FOREIGN KEY (customer_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.customers`(customer_id)
        NOT ENFORCED,

    CONSTRAINT fk_orders_shipping_city
        FOREIGN KEY (shipping_city_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.cities`(city_id)
        NOT ENFORCED
)
"""

ejecutar_ddl(sql_orders)

La tabla `order_items` es necesaria para resolver la relación _Many-to-many_.

Hay unas reglas lógicas que se tendrán en cuenta desde Python:

```python
quantity > 0
unit_price >= 0
unit_cost >= 0
0 <= discount_percent <= 100
```

In [18]:
sql_order_items = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.order_items` (
    order_item_id INT64 NOT NULL,
    order_id INT64 NOT NULL,
    product_id INT64 NOT NULL,

    quantity INT64 NOT NULL,

    unit_price NUMERIC(10, 2) NOT NULL,
    unit_cost NUMERIC(10, 2) NOT NULL,

    discount_percent NUMERIC(5, 2) DEFAULT 0.00 NOT NULL,

    PRIMARY KEY (order_item_id) NOT ENFORCED,

    CONSTRAINT fk_order_items_order
        FOREIGN KEY (order_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.orders`(order_id)
        NOT ENFORCED,

    CONSTRAINT fk_order_items_product
        FOREIGN KEY (product_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.products`(product_id)
        NOT ENFORCED
)
"""

ejecutar_ddl(sql_order_items)

En la tabla `payments` se almacenan los pagos.

De nuevo restricciones necesarias para `payment_method` y `status` se tendrán en cuenta desde Python:

```python
VALID_PAYMENT_METHODS = {
    "apple_pay",
    "bank_transfer",
    "card",
    "cash_on_delivery",
    "google_pay",
    "paypal",
    "samsung_pay",
}

VALID_STATUSES = {
    "completed",
    "failed",
    "pending",
    "refunded",
}
```

Además debe tener las siguientes reglas:

```python
amount >= 0
external_reference único
status_updated_at >= payment_date
```

In [19]:
sql_payments = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.payments` (
    payment_id INT64 NOT NULL,
    order_id INT64 NOT NULL,

    payment_method STRING NOT NULL,
    status STRING DEFAULT 'pending' NOT NULL,

    amount NUMERIC(12, 2) NOT NULL,
    payment_date DATETIME NOT NULL,
    status_updated_at DATETIME NOT NULL,

    external_reference STRING(80) NOT NULL,

    PRIMARY KEY (payment_id) NOT ENFORCED,

    CONSTRAINT fk_payments_order
        FOREIGN KEY (order_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.orders`(order_id)
        NOT ENFORCED
)
"""

ejecutar_ddl(sql_payments)

La tabla `reviews` no añade `customer_id` ni `product_id` porque sería redundante ya que se puede llegar a través de la tabla `order_items`.

In [20]:
sql_reviews = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.reviews` (
    review_id INT64 NOT NULL,
    order_item_id INT64 NOT NULL,

    rating INT64 NOT NULL,
    comment STRING(1000),
    review_date DATETIME NOT NULL,

    PRIMARY KEY (review_id) NOT ENFORCED,

    CONSTRAINT fk_reviews_order_item
        FOREIGN KEY (order_item_id)
        REFERENCES `{PROJECT_ID}.{DATASET_ID}.order_items`(order_item_id)
        NOT ENFORCED
)
"""

ejecutar_ddl(sql_reviews)

#### Comprobación de las tablas de referencia, de negocio y transaccionales

In [21]:
# Comprobación de que todas las tablas están creadas correctamente
table_names = [
    "countries",
    "cities",
    "acquisition_channels",
    "categories",
    "brands",
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "reviews",
]

for table_name in table_names:
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    table = client.get_table(table_id)

    print(f"\n--- {table_name} ---")

    for field in table.schema:
        print(
            field.name,
            "| tipo:", field.field_type,
            "| modo:", field.mode
        )


--- countries ---
country_id | tipo: INTEGER | modo: REQUIRED
country_code | tipo: STRING | modo: REQUIRED
country_name | tipo: STRING | modo: REQUIRED

--- cities ---
city_id | tipo: INTEGER | modo: REQUIRED
country_id | tipo: INTEGER | modo: REQUIRED
city_name | tipo: STRING | modo: REQUIRED

--- acquisition_channels ---
channel_id | tipo: INTEGER | modo: REQUIRED
channel_code | tipo: STRING | modo: REQUIRED
channel_name | tipo: STRING | modo: REQUIRED

--- categories ---
category_id | tipo: INTEGER | modo: REQUIRED
category_name | tipo: STRING | modo: REQUIRED
description | tipo: STRING | modo: NULLABLE

--- brands ---
brand_id | tipo: INTEGER | modo: REQUIRED
brand_name | tipo: STRING | modo: REQUIRED

--- customers ---
customer_id | tipo: INTEGER | modo: REQUIRED
first_name | tipo: STRING | modo: REQUIRED
last_name | tipo: STRING | modo: REQUIRED
email | tipo: STRING | modo: REQUIRED
phone | tipo: STRING | modo: NULLABLE
city_id | tipo: INTEGER | modo: REQUIRED
channel_id | tipo:

In [22]:
# Comprobación de que todas las PK y FK son correctas
sql_constraints = f"""
SELECT
    table_name,
    constraint_name,
    constraint_type,
    enforced
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLE_CONSTRAINTS`
ORDER BY table_name, constraint_type, constraint_name
"""

constraints = client.query(sql_constraints).to_dataframe()

constraints

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,constraint_name,constraint_type,enforced
0,acquisition_channels,acquisition_channels.pk$,PRIMARY KEY,NO
1,brands,brands.pk$,PRIMARY KEY,NO
2,categories,categories.pk$,PRIMARY KEY,NO
3,cities,cities.fk_cities_country,FOREIGN KEY,NO
4,cities,cities.pk$,PRIMARY KEY,NO
5,countries,countries.pk$,PRIMARY KEY,NO
6,customers,customers.fk_customers_channel,FOREIGN KEY,NO
7,customers,customers.fk_customers_city,FOREIGN KEY,NO
8,customers,customers.pk$,PRIMARY KEY,NO
9,order_items,order_items.fk_order_items_order,FOREIGN KEY,NO


In [23]:
# Comprobación para verificar que no falta ninguna tabla
existing_tables = {
    table.table_id for table in client.list_tables(DATASET_ID)
}

expected_tables = set(table_names)

missing_tables = expected_tables - existing_tables
unexpected_tables = existing_tables - expected_tables

print("Tablas esperadas:", len(expected_tables))
print("Tablas encontradas:", len(existing_tables))
print("Tablas que faltan:", missing_tables)
print("Tablas no esperadas:", unexpected_tables)

Tablas esperadas: 11
Tablas encontradas: 11
Tablas que faltan: set()
Tablas no esperadas: set()
